# Probe & baseline results

Reads whatever is in `runs/`. Loads no model and needs no GPU -- this is for
looking at results, not producing them.

The question behind everything here: **does the model know enough chess for
RL to have anything to amplify?** RL boosts responses a model already gives
sometimes; it cannot install a missing capability.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

from plot import GRID, INK, INK_SOFT, SERIES, SURFACE, style

RUNS = Path("runs")
for path in sorted(RUNS.glob("*.json")):
    print(f"{path.name:30s} {path.stat().st_size / 1024:6.0f} KB")

## 1. Capability probe

`moves` -- where can a lone piece go on an empty board (does it know the rules)  
`board` -- what sits on a square after this movetext (can it track state)

The dashed line on `board` is the score of always answering "empty". The probe
is built 50/50 occupied/empty, so **anything at or below that line is not
tracking the board** -- it is guessing.

In [ ]:
probe = json.load(open(RUNS / "probe.json"))
kinds = ["moves", "board"]
scores = {
    k: [i["score"] for i in probe if i["kind"] == k] for k in kinds
}

fig, ax = plt.subplots(figsize=(6, 2.6), facecolor=SURFACE)
pct = [100 * sum(scores[k]) / len(scores[k]) for k in kinds]
ax.barh(kinds[::-1], pct[::-1], height=0.5, color=SERIES["model"], zorder=3)
for y, value in enumerate(pct[::-1]):
    ax.annotate(f"{value:.0f}%", xy=(value, y), xytext=(6, 0),
                textcoords="offset points", va="center", color=INK, fontsize=9)

ax.axvline(50, color=INK_SOFT, linestyle=":", linewidth=1)
ax.annotate('always say "empty"', xy=(50, 1.0), xytext=(4, 0),
            textcoords="offset points", color=INK_SOFT, fontsize=8)
ax.set_xlim(0, 105)
ax.set_xlabel("% correct", color=INK_SOFT, fontsize=9)
ax.set_title(f"Capability probe (n={len(scores['moves'])} each)",
             color=INK, fontsize=11, loc="left", pad=10)
style(ax)
ax.grid(axis="y", visible=False)
plt.tight_layout()

### What it got wrong, and how

The *shape* of the errors matters more than the score. Listing squares next to
a piece rather than along its lines means it has no movement rule; naming a
square outside a1-h8 means it has no board.

In [ ]:
import re

SQUARE_ANY = re.compile(r"\b[a-h]\d+\b")

moves = [i for i in probe if i["kind"] == "moves"]
impossible = [
    s for i in moves for s in SQUARE_ANY.findall(i["said"].lower())
    if not s[1:].isdigit() or not 1 <= int(s[1:]) <= 8
]
print(f"squares named that do not exist on a chessboard: {len(impossible)}")
print(f"  e.g. {sorted(set(impossible))[:12]}")

print()
for item in moves[:6]:
    piece = item["prompt"].split("white ")[1].split(" on")[0]
    square = item["prompt"].split(" on ")[1].split(".")[0]
    print(f"{piece:7s} on {square}")
    print(f"   want {' '.join(item['answer'])}")
    print(f"   got  {item['said'][:70]}")

## 2. Baseline runs

`answered` counts a `\boxed{...}` of any kind. **A boxed value that is not
move-shaped is not an attempt** -- the model echoing the prompt's own
placeholder inflated this badly in the first no-think run.

In [ ]:
SAN = re.compile(r"^(O-O-O|O-O|[KQRBN]?[a-h]?[1-8]?x?[a-h][1-8](=[QRBN])?[+#]?)$")


def load_run(name):
    return json.load(open(RUNS / name))


def model_arm(results):
    return next(a for a in results
                if not a["name"].startswith(("random", "stockfish")))


def breakdown(name):
    rows = model_arm(load_run(name))["rows"]
    n = len(rows)
    legal = [r for r in rows if r["legal"]]
    boxed_junk = [r for r in rows if r["answered"] and not r["legal"]
                  and not SAN.match((r["raw"] or "").strip())]
    real_illegal = [r for r in rows if r["answered"] and not r["legal"]
                    and SAN.match((r["raw"] or "").strip())]
    silent = [r for r in rows if not r["answered"]]

    print(f"{name}  (n={n})")
    print(f"  legal move          {len(legal):4d}")
    print(f"  illegal move named  {len(real_illegal):4d}")
    print(f"  boxed non-move      {len(boxed_junk):4d}   {Counter((r['raw'] or '') for r in boxed_junk).most_common(3)}")
    print(f"  no answer           {len(silent):4d}")
    attempts = len(legal) + len(real_illegal)
    if attempts:
        print(f"  -> of {attempts} real attempts, {len(legal) / attempts:.0%} legal")
    return rows


for path in sorted(RUNS.glob("*.json")):
    if path.name == "probe.json":
        continue
    breakdown(path.name)
    print()

## 3. Move quality vs the references

Only over moves that were actually legal. Below the random line means the
model's *choices* are worse than chance -- plausible-looking moves picked
without reference to the position.

In [ ]:
import statistics as st

for path in sorted(RUNS.glob("*.json")):
    if path.name == "probe.json":
        continue
    results = load_run(path.name)
    print(path.name)
    for arm in results:
        losses = [r["cp_loss"] for r in arm["rows"] if r["cp_loss"] is not None]
        if losses:
            good = sum(x <= 50 for x in losses) / len(losses)
            print(f"   {arm['name'][:38]:40s} n={len(losses):3d}  "
                  f"mean {st.mean(losses):5.0f}  median {st.median(losses):5.0f}  "
                  f"good {good:5.1%}")
        else:
            print(f"   {arm['name'][:38]:40s} no legal moves")
    print()

## 4. Degenerate-policy check

A published GRPO run on 8B models converged on pushing the a-pawn >80% of the
time: nearly always legal, rarely catastrophic, and the best constant answer
when you cannot read a board. Mean cp_loss improves the whole way. **This is
the panel that separates learning from collapse.**

In [ ]:
for path in sorted(RUNS.glob("*.json")):
    if path.name == "probe.json":
        continue
    rows = model_arm(load_run(path.name))["rows"]
    played = [r["move"] for r in rows if r.get("move")]
    if not played:
        print(f"{path.name}: no legal moves\n")
        continue
    counts = Counter(played).most_common(8)
    top, n_top = counts[0]
    share = n_top / len(played)
    flag = "  <- DEGENERATE" if share >= 0.20 else ""
    print(f"{path.name}: {len(played)} legal, top {top} at {share:.0%}{flag}")
    print(f"   {counts}\n")

## 5. Read the completions

Numbers say a run failed; only the text says why.

In [ ]:
RUN = "movetext-nothink.json"  # change to inspect another run
INDEX = 0

row = model_arm(load_run(RUN))["rows"][INDEX]
print(f"fen   {row['fen']}")
print(f"raw   {row['raw']!r}")
print(f"move  {row['move']}   legal={row['legal']}  cp_loss={row['cp_loss']}")
print(f"tokens {row['tokens']}  truncated={row['truncated']}")
print("\n--- completion ---")
print(row["completion"][:1500])